In [16]:
import os
import time
import cv2
import torch
import torchvision
from yolox.exp import get_exp
from yolox.utils import fuse_model, get_model_info, vis
from yolox.data.data_augment import ValTransform
from yolox.data.datasets import COCO_CLASSES

# --- Configuration ---
exp_file = "exps/default/yolox_nano_cid.py"
ckpt_file = "YOLOX_outputs/yolox_nano_cid/best_ckpt.pth"
img_path = (
    "datasets/your_dataset/val2017/0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg"
)
conf_thre = 0.25
nms_thre = 0.45
test_size = (320, 320)
device = "cuda" if torch.cuda.is_available() else "cpu"
save_result = True

# --- Load Experiment and Model ---
exp = get_exp(exp_file, None)
exp.test_conf = conf_thre
exp.nmsthre = nms_thre
exp.test_size = test_size

model = exp.get_model()
ckpt = torch.load(ckpt_file, map_location="cpu")
model.load_state_dict(ckpt["model"])
model.eval()
if device == "cuda":
    model.cuda()


def postprocess(
    prediction, num_classes, conf_thre=0.7, nms_thre=0.45, class_agnostic=False
):
    box_corner = prediction.new(prediction.shape)
    box_corner[:, :, 0] = prediction[:, :, 0] - prediction[:, :, 2] / 2
    box_corner[:, :, 1] = prediction[:, :, 1] - prediction[:, :, 3] / 2
    box_corner[:, :, 2] = prediction[:, :, 0] + prediction[:, :, 2] / 2
    box_corner[:, :, 3] = prediction[:, :, 1] + prediction[:, :, 3] / 2
    prediction[:, :, :4] = box_corner[:, :, :4]

    print("prediction:", prediction[0][0])
    print("prediction:", prediction[0][1])
    print("prediction:", prediction[0][2])

    output = [None for _ in range(len(prediction))]
    for i, image_pred in enumerate(prediction):
        print("image_pred shape:", image_pred.shape)

        # If none are remaining => process next image
        if not image_pred.size(0):
            continue
        # Get score and class with highest confidence
        class_conf, class_pred = torch.max(
            image_pred[:, 5 : 5 + num_classes], 1, keepdim=True
        )
        print("image_pred:", image_pred[0, 5 : 5 + num_classes])
        print("class_conf shape:", class_conf.shape)
        print("class_conf sample:", class_conf[0])
        print("class_conf sample:", class_conf[1])
        print("class_pred shape:", class_pred.shape)
        print("class_pred sample:", class_pred[0])
        print("class_pred sample:", class_pred[1])

        print("class_conf.squeeze(): ", class_conf.squeeze())

        conf_mask = (image_pred[:, 4] * class_conf.squeeze() >= conf_thre).squeeze()
        print("conf_mask shape:", conf_mask.shape)
        print("conf_mask:", conf_mask)
        print("conf_mask True:", torch.sum(conf_mask).item())
        
        # Detections ordered as (x1, y1, x2, y2, obj_conf, class_conf, class_pred)
        detections = torch.cat((image_pred[:, :5], class_conf, class_pred.float()), 1)
        print("detections sample:", detections[2000])
        detections = detections[conf_mask]
        print("detections shape:", detections.shape)
        print("detections:", detections)
        if not detections.size(0):
            continue

        if class_agnostic:
            nms_out_index = torchvision.ops.nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                nms_thre,
            )
        else:
            nms_out_index = torchvision.ops.batched_nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                detections[:, 6],
                nms_thre,
            )

        detections = detections[nms_out_index]
        if output[i] is None:
            output[i] = detections
        else:
            output[i] = torch.cat((output[i], detections))

    print("Final output shape:", len(output), "tensors")
    print("Output example:", output[0] if output else "No detections")
    return output


# --- Predictor ---
class Predictor:
    def __init__(self, model, exp, device):
        self.model = model
        self.num_classes = exp.num_classes
        print(f"Model: {exp.exp_name}, Classes: {self.num_classes}")
        self.confthre = exp.test_conf
        self.nmsthre = exp.nmsthre
        self.test_size = exp.test_size
        self.device = device
        self.preproc = ValTransform()
        self.cls_names = COCO_CLASSES

    def inference(self, img_path):
        img = cv2.imread(img_path)
        ratio = min(self.test_size[0] / img.shape[0], self.test_size[1] / img.shape[1])
        img_info = {
            "height": img.shape[0],
            "width": img.shape[1],
            "raw_img": img,
            "ratio": ratio,
            "file_name": os.path.basename(img_path),
        }
        img, _ = self.preproc(img, None, self.test_size)
        img = torch.from_numpy(img).unsqueeze(0).float()
        print(f"Input image shape: {img.shape}, Ratio: {ratio:.2f}")
        if self.device == "cuda":
            img = img.cuda()
        with torch.no_grad():
            outputs = self.model(img)
            print(f"Model outputs shape: {outputs.shape}")
            print(f"Model outputs: {len(outputs)} tensors")
            outputs = postprocess(
                outputs,
                self.num_classes,
                self.confthre,
                self.nmsthre,
                class_agnostic=True,
            )
            print(f"Postprocessed outputs: {outputs}")
        return outputs, img_info

    def visual(self, output, img_info, cls_conf=0.35):
        ratio = img_info["ratio"]
        img = img_info["raw_img"]
        if output is None:
            return img
        output = output.cpu()
        print(f"Output shape: {output.shape}")
        bboxes = output[:, 0:4] / ratio
        cls = output[:, 6]
        scores = output[:, 4] * output[:, 5]
        valid_mask = (cls >= 0) & (cls < len(self.cls_names))
        bboxes = bboxes[valid_mask]
        cls = cls[valid_mask]
        scores = scores[valid_mask]
        print(f"Detected {len(bboxes)} objects.")
        print(
            f"Image: {img_info['file_name']}, Size: {img_info['width']}x{img_info['height']}, Ratio: {ratio:.2f}"
        )
        print(f"Classes: {', '.join([self.cls_names[int(c)] for c in cls])}")
        print(f"Scores: {', '.join([f'{s:.6f}' for s in scores])}")
        return vis(img, bboxes, scores, cls, cls_conf, self.cls_names)


predictor = Predictor(model, exp, device)
outputs, img_info = predictor.inference(img_path)
print("outputs:", outputs)
result_img = predictor.visual(outputs[0], img_info, conf_thre)

if save_result:
    save_dir = "yolox_outputs/vis_res"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, os.path.basename(img_path))
    cv2.imwrite(save_path, result_img)
    print(f"Saved result to {save_path}")
else:
    cv2.imshow("YOLOX Result", result_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

C:\Users\Wave\AppData\Local\Temp\ipykernel_28732\3890088483.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cpu")


Model: yolox_nano_cid, Classes: 2
Input image shape: torch.Size([1, 3, 320, 320]), Ratio: 1.00
Model outputs shape: torch.Size([1, 2100, 7])
Model outputs: 1 tensors
prediction: tensor([-9.0939e-01, -3.2014e+00,  7.1931e+00,  5.9564e+00,  5.1193e-06,
         1.1042e-02,  1.1416e-02], device='cuda:0')
prediction: tensor([ 7.8577e+00, -3.1574e+00,  1.5852e+01,  6.3405e+00,  5.1770e-07,
         1.0346e-02,  1.2374e-02], device='cuda:0')
prediction: tensor([ 1.5928e+01, -3.0735e+00,  2.3825e+01,  6.2713e+00,  7.4262e-07,
         1.0929e-02,  1.0568e-02], device='cuda:0')
image_pred shape: torch.Size([2100, 7])
image_pred: tensor([0.0110, 0.0114], device='cuda:0')
class_conf shape: torch.Size([2100, 1])
class_conf sample: tensor([0.0114], device='cuda:0')
class_conf sample: tensor([0.0124], device='cuda:0')
class_pred shape: torch.Size([2100, 1])
class_pred sample: tensor([1], device='cuda:0')
class_pred sample: tensor([1], device='cuda:0')
class_conf.squeeze():  tensor([0.0114, 0.0124, 

In [66]:
import os
import time
import cv2
import torch
from yolox.utils import fuse_model, get_model_info, postprocess, vis
from yolox.data.data_augment import ValTransform
from yolox.data.datasets import COCO_CLASSES

# --- Configuration ---
ckpt_file = "YOLOX_outputs/yolox_nano_cid/best_ckpt.pth"
img_path = (
    "datasets/your_dataset/val2017/0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg"
)
conf_thre = 0.25
nms_thre = 0.45
test_size = (320, 320)
device = "cuda" if torch.cuda.is_available() else "cpu"
save_result = True


# --- Define Your Model ---
class YourModelClass(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Define your layers here
        # Example: self.conv = torch.nn.Conv2d(3, 16, 3, 1, 1)
        pass

    def forward(self, x):
        # Implement your forward pass
        # Example: return self.conv(x)
        pass


model = YourModelClass()
ckpt = torch.load(ckpt_file, map_location="cpu")
model.load_state_dict(ckpt["model"])
model.eval()
model.to(device)


# --- Predictor ---
class Predictor:
    def __init__(self, model, device):
        self.model = model
        self.num_classes = 2
        print(f"Classes: {self.num_classes}")
        self.confthre = conf_thre
        self.nmsthre = nms_thre
        self.test_size = test_size
        self.device = device
        self.preproc = ValTransform()
        self.cls_names = COCO_CLASSES

    def inference(self, img_path):
        img = cv2.imread(img_path)
        ratio = min(self.test_size[0] / img.shape[0], self.test_size[1] / img.shape[1])
        img_info = {
            "height": img.shape[0],
            "width": img.shape[1],
            "raw_img": img,
            "ratio": ratio,
            "file_name": os.path.basename(img_path),
        }
        img, _ = self.preproc(img, None, self.test_size)
        img = torch.from_numpy(img).unsqueeze(0).float()
        print(f"Input image shape: {img.shape}, Ratio: {ratio:.2f}")
        if self.device == "cuda":
            img = img.cuda()
        with torch.no_grad():
            outputs = self.model(img)
            outputs = postprocess(
                outputs,
                self.num_classes,
                self.confthre,
                self.nmsthre,
                class_agnostic=True,
            )
        return outputs, img_info

    def visual(self, output, img_info, cls_conf=0.35):
        ratio = img_info["ratio"]
        img = img_info["raw_img"]
        if output is None:
            return img
        output = output.cpu()
        print(f"Output shape: {output.shape}")
        bboxes = output[:, 0:4] / ratio
        cls = output[:, 6]
        scores = output[:, 4] * output[:, 5]
        valid_mask = (cls >= 0) & (cls < len(self.cls_names))
        bboxes = bboxes[valid_mask]
        cls = cls[valid_mask]
        scores = scores[valid_mask]
        print(f"Detected {len(bboxes)} objects.")
        print(
            f"Image: {img_info['file_name']}, Size: {img_info['width']}x{img_info['height']}, Ratio: {ratio:.2f}"
        )
        print(f"Classes: {', '.join([self.cls_names[int(c)] for c in cls])}")
        print(f"Scores: {', '.join([f'{s:.2f}' for s in scores])}")
        return vis(img, bboxes, scores, cls, cls_conf, self.cls_names)


predictor = Predictor(model, device)
outputs, img_info = predictor.inference(img_path)
print("outputs:", outputs)
result_img = predictor.visual(outputs[0], img_info, conf_thre)

if save_result:
    save_dir = "yolox_outputs/vis_res"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, os.path.basename(img_path))
    cv2.imwrite(save_path, result_img)
    print(f"Saved result to {save_path}")
else:
    cv2.imshow("YOLOX Result", result_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

C:\Users\Wave\AppData\Local\Temp\ipykernel_11580\2020398878.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cpu")


RuntimeError: Error(s) in loading state_dict for YourModelClass:
	Unexpected key(s) in state_dict: "backbone.backbone.stem.conv.conv.weight", "backbone.backbone.stem.conv.bn.weight", "backbone.backbone.stem.conv.bn.bias", "backbone.backbone.stem.conv.bn.running_mean", "backbone.backbone.stem.conv.bn.running_var", "backbone.backbone.stem.conv.bn.num_batches_tracked", "backbone.backbone.dark2.0.dconv.conv.weight", "backbone.backbone.dark2.0.dconv.bn.weight", "backbone.backbone.dark2.0.dconv.bn.bias", "backbone.backbone.dark2.0.dconv.bn.running_mean", "backbone.backbone.dark2.0.dconv.bn.running_var", "backbone.backbone.dark2.0.dconv.bn.num_batches_tracked", "backbone.backbone.dark2.0.pconv.conv.weight", "backbone.backbone.dark2.0.pconv.bn.weight", "backbone.backbone.dark2.0.pconv.bn.bias", "backbone.backbone.dark2.0.pconv.bn.running_mean", "backbone.backbone.dark2.0.pconv.bn.running_var", "backbone.backbone.dark2.0.pconv.bn.num_batches_tracked", "backbone.backbone.dark2.1.conv1.conv.weigh

In [36]:
!python tools/demo.py image -n yolox-s -c weights/yolox_s.pth --path assets/dog.jpg --conf 0.25 --nms 0.45 --tsize 640 --save_result --device gpu


ckpt keys: dict_keys(['start_epoch', 'model', 'optimizer', 'amp'])


2025-07-12 13:31:53.518 | INFO     | __main__:main:267 - Args: Namespace(demo='image', experiment_name='yolox_s', name='yolox-s', path='assets/dog.jpg', camid=0, save_result=True, exp_file=None, ckpt='weights/yolox_s.pth', device='gpu', conf=0.25, nms=0.45, tsize=640, fp16=False, legacy=False, fuse=False, trt=False)
2025-07-12 13:31:53.910 | INFO     | __main__:main:277 - Model Summary: Params: 8.97M, Gflops: 26.93
2025-07-12 13:31:54.113 | INFO     | __main__:main:290 - loading checkpoint
d:\git\YOLOX-custom\tools\demo.py:291: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that coul

In [40]:
!python tools/demo.py image -f exps/default/yolox_nano_cid.py -c YOLOX_outputs/yolox_nano_cid/best_ckpt.pth --path datasets/your_dataset/val2017/0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg --conf 0.25 --nms 0.45 --tsize 320 --save_result --device gpu


ckpt keys: dict_keys(['start_epoch', 'model', 'optimizer', 'best_ap', 'curr_ap'])


2025-07-12 13:36:40.241 | INFO     | __main__:main:267 - Args: Namespace(demo='image', experiment_name='yolox_nano_cid', name=None, path='datasets/your_dataset/val2017/0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg', camid=0, save_result=True, exp_file='exps/default/yolox_nano_cid.py', ckpt='YOLOX_outputs/yolox_nano_cid/best_ckpt.pth', device='gpu', conf=0.25, nms=0.45, tsize=320, fp16=False, legacy=False, fuse=False, trt=False)
2025-07-12 13:36:40.617 | INFO     | __main__:main:277 - Model Summary: Params: 0.90M, Gflops: 0.64
2025-07-12 13:36:40.976 | INFO     | __main__:main:290 - loading checkpoint
d:\git\YOLOX-custom\tools\demo.py:291: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a 